# 07 — Convolution Kernels

Covers Phase 7 of the kernel roadmap:
- `conv1d` — direct 1D convolution, baseline
- `conv2d` — direct 2D convolution (sliding window, NCHW layout)
- `depthwise_conv2d` — depthwise 2D (each channel independently)

**Metric**: TFLOPS — compute-bound at large channel count  
`(2 × B × C_out × N_out × C_in × K × 1e-12) / (ms × 1e-3)` for 1D  
`(2 × B × C_out × H_out × W_out × C_in × K² × 1e-12) / (ms × 1e-3)` for 2D

**References**:
- Triton conv2d tutorial: https://triton-lang.org/main/getting-started/tutorials/
- Depthwise convolution (MobileNet): https://arxiv.org/abs/1704.04861

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/Bhavikupadhyay/triton-kernels.git"
REPO_BRANCH = "feature/conv1d"
REPO_DIR    = "/content/drive/MyDrive/triton-kernels"

if os.path.exists(REPO_DIR):
    os.system(f"git -C {REPO_DIR} fetch --all")
    os.system(f"git -C {REPO_DIR} checkout -f {REPO_BRANCH}")
    os.system(f"git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}")
else:
    os.system(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
import subprocess
subprocess.run(["bash", "scripts/setup_colab.sh"])
import torch
assert torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import torch
import torch.nn.functional as F
import triton

from kernels.convolution.conv1d import conv1d, test_conv1d, benchmark_conv1d
# Uncomment as kernels are implemented:
# from kernels.convolution.conv2d import conv2d, test_conv2d, benchmark_conv2d
# from kernels.convolution.depthwise_conv2d import depthwise_conv2d, test_depthwise_conv2d, benchmark_depthwise_conv2d

print("Imports ready")

## 1. conv1d

**File**: `kernels/convolution/conv1d.py`  
**PyTorch equivalent**: `F.conv1d(x, weight, padding=0)`  
**Layout**: `(B, C_in, N)` input, `(C_out, C_in, K)` weight, `(B, C_out, N_out)` output  
**Key**: Each output element is a dot product over `C_in × K`. One program per `(B, C_out, tile)`.  
**Complexity**: `O(B × C_out × N_out × C_in × K)` compute, `O(B × C_in × N + C_out × C_in × K)` memory

In [ ]:
# ── conv1d: Correctness ──────────────────────────────────────────────────────
test_conv1d()

In [ ]:
# ── conv1d: Benchmark ────────────────────────────────────────────────────────
import os
os.makedirs("benchmarks/results/convolution", exist_ok=True)
benchmark_conv1d.run(
    print_data=True,
    show_plots=True,
    save_path="benchmarks/results/convolution",
)

**Interpretation**: Fill in after Colab run.

## 2. conv2d

**File**: `kernels/convolution/conv2d.py`  
**PyTorch equivalent**: `F.conv2d(x, weight, padding=0)`  
**Layout**: `(B, C_in, H, W)` input (NCHW), `(C_out, C_in, K, K)` weight  
**Key**: Extends conv1d to 2D — the inner loop is over `C_in × K × K`. Tiled along the output spatial dimension.  
**Complexity**: `O(B × C_out × H_out × W_out × C_in × K²)` compute

In [ ]:
# ── conv2d: Correctness ──────────────────────────────────────────────────────
# test_conv2d()

In [ ]:
# ── conv2d: Benchmark ────────────────────────────────────────────────────────
# benchmark_conv2d.run(
#     print_data=True,
#     show_plots=True,
#     save_path="benchmarks/results/convolution",
# )

**Interpretation**: Fill in after Colab run.

## 3. depthwise_conv2d

**File**: `kernels/convolution/depthwise_conv2d.py`  
**PyTorch equivalent**: `F.conv2d(x, weight, groups=C_in)`  
**Layout**: `(B, C, H, W)` input, `(C, 1, K, K)` weight — one filter per channel, no cross-channel mixing  
**Key**: `groups=C_in` in standard conv. Each program handles one `(batch, channel, output_tile)`. Reduced compute but same HBM access → memory-bound.  
**Complexity**: `O(B × C × H_out × W_out × K²)` compute — `C_in × C_out` factor removed vs conv2d

In [ ]:
# ── depthwise_conv2d: Correctness ────────────────────────────────────────────
# test_depthwise_conv2d()

In [ ]:
# ── depthwise_conv2d: Benchmark ──────────────────────────────────────────────
# benchmark_depthwise_conv2d.run(
#     print_data=True,
#     show_plots=True,
#     save_path="benchmarks/results/convolution",
# )

**Interpretation**: Fill in after Colab run.

In [ ]:
# ── Summary Table ────────────────────────────────────────────────────────────
# import pandas as pd, glob
# csvs = glob.glob("benchmarks/results/convolution/*.csv")
# if csvs:
#     print(pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True).to_string(index=False))
# else:
#     print("No CSVs yet.")